# Point-in-time rolling betas and beta-decile sorts (CRSP monthly)

**What this notebook does**
1. Loads a CRSP monthly stock panel and a market return series (S&P 500 recommended).
2. Estimates each stock's market beta with rolling OLS over 6m / 1y / 2y / 5y / 10y windows.
3. Sorts stocks into beta deciles and measures next-month returns.

**No look-ahead:** the beta dated month-end *t* uses only returns from months *t-W+1 ... t*. Stocks are sorted on it at *t* and held during *t+1*.

**1-month beta** cannot be estimated from monthly data (one observation). It needs daily returns, see the last section.

## 1. Imports

In [8]:
import os
import warnings
import numpy as np
import pandas as pd

## 2. Configuration: edit this cell only

| Setting | Meaning |
|---|---|
| `CRSP_PATH` | CRSP monthly parquet (permno, date, ret, mktcap, exchcd, shrcd, prc, ...) |
| `MARKET_PATH` | Parquet/CSV with columns `date`, `mkt_ret` and optionally `rf` (decimal monthly returns). Set to `None` to use a value-weighted CRSP proxy (**not** SPX). |
| `OUT_DIR` | Folder for saved results |
| `NYSE_BREAKS` | `True`: decile cutoffs from NYSE stocks only (`exchcd == 1`) |
| `MIN_PRICE` | Exclude stocks priced below this at formation from the sorts (0 = no filter) |
| `WINDOWS` / `MIN_OBS` | Window length in months, and minimum valid months required to report a beta |

In [9]:
CRSP_PATH   = "/Users/daniilg/Downloads/crsp_monthly_1963_2026.parquet"
MARKET_PATH = None            # e.g. "spx_monthly.parquet"  (columns: date, mkt_ret, rf)
OUT_DIR     = "beta_out"
NYSE_BREAKS = False
MIN_PRICE   = 0.0

WINDOWS = {"6m": 6, "1y": 12, "2y": 24, "5y": 60, "10y": 120}
MIN_OBS = {"6m": 6, "1y": 12, "2y": 24, "5y": 36, "10y": 60}

os.makedirs(OUT_DIR, exist_ok=True)

## 3. Functions

### 3a. Data loading

In [10]:
def load_crsp(path, min_price=0.0):
    """Read the CRSP panel, keep common stocks, snap dates to month-end."""
    cols = ["permno", "date", "ret", "prc", "mktcap", "exchcd", "shrcd"]
    df = pd.read_parquet(path, columns=cols)
    df = df[df.shrcd.isin([10, 11])].copy()          # common stocks only
    df["date"] = df["date"] + pd.offsets.MonthEnd(0)
    df["prc"] = df["prc"].abs()                      # CRSP stores bid/ask midpoints as negative prices
    if min_price > 0:                                # low-priced stocks get no weight in the sort
        df.loc[df.prc < min_price, "mktcap"] = np.nan
    return df


def build_market(df, market_path):
    """Return (market return, risk-free rate, label), both indexed by month-end."""
    if market_path:
        m = pd.read_parquet(market_path) if market_path.endswith(".parquet") else pd.read_csv(market_path)
        m["date"] = pd.to_datetime(m["date"]) + pd.offsets.MonthEnd(0)
        m = m.set_index("date").sort_index()
        rf = m["rf"] if "rf" in m else pd.Series(0.0, index=m.index)
        return m["mkt_ret"], rf, "user-supplied market"

    # PROXY: value-weighted CRSP market using prior-month market cap as weights
    d = df.dropna(subset=["ret"]).sort_values(["permno", "date"]).copy()
    d["w"] = d.groupby("permno")["mktcap"].shift(1)
    d = d.dropna(subset=["w"])
    d["rw"] = d.ret * d.w
    agg = d.groupby("date")[["rw", "w"]].sum()
    mkt = agg.rw / agg.w
    warnings.warn("No market file given: using VW CRSP proxy, NOT the S&P 500.")
    return mkt, pd.Series(0.0, index=mkt.index), "VW CRSP proxy (not SPX)"

### 3b. Rolling beta

In [11]:
def rolling_beta(R, m, window, min_obs):
    """Rolling OLS slope of each stock's excess return on the market excess return.

    R : DataFrame, date x permno, excess returns (NaN = missing)
    m : Series, market excess return, indexed by date

    beta = Cov(r, m) / Var(m), computed from rolling sums over months where BOTH
    the stock and the market are observed. Fully vectorised across all stocks.
    """
    mm = m.reindex(R.index)
    V  = (R.notna() & mm.notna().values[:, None]).astype("float32")   # 1 if observation is valid
    Rz = R.fillna(0.0).astype("float32")
    mv = mm.fillna(0.0).astype("float32").values[:, None]

    roll = lambda x: x.rolling(window, min_periods=1).sum()
    n   = roll(V)                 # number of valid months in window
    Sr  = roll(Rz * V)            # sum r
    Sm  = roll(V * mv)            # sum m
    Srm = roll(Rz * mv * V)       # sum r*m
    Smm = roll(V * mv * mv)       # sum m^2

    cov  = Srm - Sr * Sm / n
    var  = Smm - Sm * Sm / n
    beta = cov / var
    return beta.where(n >= min_obs)

### 3c. Decile sort

In [12]:
def decile_sort(beta, ret_next, mcap, exch, nyse_breaks=False, min_stocks=100):
    """Form beta deciles each month and return next-month portfolio returns.

    beta, mcap, exch : date x permno, known at formation month t
    ret_next         : date x permno, returns in month t+1 (i.e. R.shift(-1))
    Output is indexed by the HOLDING month (t+1), columns EW_1..EW_10, VW_1..VW_10, *_HL.
    Decile 1 = lowest beta, decile 10 = highest. VW weights = market cap at t.
    """
    rows = []
    for t in beta.index[:-1]:
        b  = beta.loc[t]
        ok = b.notna() & ret_next.loc[t].notna() & mcap.loc[t].notna()
        if ok.sum() < min_stocks:
            continue
        b, r, w = b[ok], ret_next.loc[t][ok], mcap.loc[t][ok]

        # breakpoints (optionally from NYSE stocks only)
        ref = b[exch.loc[t][ok] == 1] if nyse_breaks else b
        if len(ref) < 50:
            ref = b
        cuts = np.unique(np.nanpercentile(ref, np.linspace(0, 100, 11)))
        if len(cuts) < 11:
            continue
        cuts[0], cuts[-1] = -np.inf, np.inf
        dec = pd.cut(b, cuts, labels=False, include_lowest=True) + 1

        g = pd.DataFrame({"dec": dec, "r": r, "w": w})
        ew = g.groupby("dec").r.mean()
        g["rw"] = g.r * g.w
        agg = g.groupby("dec")[["rw", "w"]].sum()
        vw = agg.rw / agg.w

        rec = {"date": ret_next.index[ret_next.index.get_loc(t) + 1]}
        for k in range(1, 11):
            rec[f"EW_{k}"] = ew.get(k, np.nan)
            rec[f"VW_{k}"] = vw.get(k, np.nan)
        rows.append(rec)

    out = pd.DataFrame(rows).set_index("date")
    for w in ("EW", "VW"):
        out[f"{w}_HL"] = out[f"{w}_10"] - out[f"{w}_1"]    # high-minus-low (zero-cost)
    return out

### 3d. Portfolio performance summary

In [13]:
def summarize(port, mkt_ex, rf):
    """For each portfolio: mean excess return, CAPM alpha, realized beta, Sharpe.

    Long-short (*_HL) columns are already excess returns, so rf is not subtracted.
    t-stats are plain OLS (no Newey-West).
    """
    rows = []
    for col in port.columns:
        sub = 0 if col.endswith("HL") else rf.reindex(port.index).fillna(0)
        x  = (port[col] - sub).dropna()
        mk = mkt_ex.reindex(x.index)

        X = np.column_stack([np.ones(len(x)), mk.values])
        coef, *_ = np.linalg.lstsq(X, x.values, rcond=None)
        res = x.values - X @ coef
        se  = np.sqrt(np.diag(res @ res / (len(x) - 2) * np.linalg.inv(X.T @ X)))

        rows.append({"portfolio": col,
                     "mean_ex_ret_%/mo": 100 * x.mean(),
                     "t_mean": x.mean() / (x.std() / np.sqrt(len(x))),
                     "alpha_%/mo": 100 * coef[0],
                     "t_alpha": coef[0] / se[0],
                     "realized_beta": coef[1],
                     "ann_sharpe": np.sqrt(12) * x.mean() / x.std(),
                     "n_months": len(x)})
    return pd.DataFrame(rows).set_index("portfolio")

## 4. Load data and build the market series

In [14]:
df = load_crsp(CRSP_PATH, MIN_PRICE)
mkt, rf, label = build_market(df, MARKET_PATH)
print("Market used:", label)
print("Stocks:", df.permno.nunique(), "| months:", df.date.nunique())

Market used: VW CRSP proxy (not SPX)
Stocks: 26519 | months: 765


/var/folders/1q/0zbdb61d5n56pdv3gl6gwmpw0000gn/T/ipykernel_4232/444600867.py:29: UserWarning: No market file given: using VW CRSP proxy, NOT the S&P 500.
  warnings.warn("No market file given: using VW CRSP proxy, NOT the S&P 500.")


## 5. Reshape to date x stock matrices

Calendar months with no CRSP record become NaN, so a window always means *calendar* months.

In [15]:
R    = df.pivot(index="date", columns="permno", values="ret")
idx  = pd.date_range(R.index.min(), R.index.max(), freq="ME")   # use "M" on pandas < 2.2

R    = R.reindex(idx)
mcap = df.pivot(index="date", columns="permno", values="mktcap").reindex(idx)
exch = df.pivot(index="date", columns="permno", values="exchcd").reindex(idx)

rf     = rf.reindex(idx).fillna(0.0)
mkt    = mkt.reindex(idx)
Rex    = R.sub(rf, axis=0)      # stock excess returns
mkt_ex = mkt - rf               # market excess returns

## 6. Estimate betas for every window

`betas["5y"]` is a date x permno DataFrame; the value at row *t* is known at the end of month *t*.

In [16]:
betas = {}
for name, w in WINDOWS.items():
    betas[name] = rolling_beta(Rex, mkt_ex, w, MIN_OBS[name])
    print(f"{name:>3}: mean cross-sectional beta = {betas[name].mean(axis=1).mean():.3f}")

# long format: one row per (date, permno), one column per window
beta_long = pd.concat({k: v.stack() for k, v in betas.items()}, axis=1)
beta_long.columns = [f"beta_{k}" for k in beta_long.columns]
beta_long.to_parquet(f"{OUT_DIR}/betas_all.parquet")
beta_long.tail()

 6m: mean cross-sectional beta = 1.096
 1y: mean cross-sectional beta = 1.134
 2y: mean cross-sectional beta = 1.148
 5y: mean cross-sectional beta = 1.138
10y: mean cross-sectional beta = 1.134


beta_6m  beta_1y  beta_2y  beta_5y  beta_10y
           permno                                              
2026-09-30 93150       NaN      NaN      NaN      NaN  1.410474
           93175       NaN      NaN      NaN      NaN  2.113030
           93301       NaN      NaN      NaN      NaN  1.030032
           93373       NaN      NaN      NaN      NaN  1.731009
           93423       NaN      NaN      NaN      NaN  2.039402

## 7. Decile sorts and performance

Decile 1 = lowest beta, 10 = highest. `HL` = decile 10 minus decile 1.

In [17]:
sorts, summaries = {}, {}
for name in WINDOWS:
    sorts[name]     = decile_sort(betas[name], R.shift(-1), mcap, exch, NYSE_BREAKS)
    summaries[name] = summarize(sorts[name], mkt_ex, rf)
    sorts[name].to_csv(f"{OUT_DIR}/decile_returns_{name}.csv")
    summaries[name].to_csv(f"{OUT_DIR}/decile_summary_{name}.csv")

In [18]:
vw_rows = [f"VW_{k}" for k in range(1, 11)] + ["VW_HL"]
for name in WINDOWS:
    print(f"\n=== Value-weighted deciles sorted on {name} beta ===")
    display(summaries[name].loc[vw_rows].round(3))


=== Value-weighted deciles sorted on 6m beta ===


,mean_ex_ret_%/mo,t_mean,alpha_%/mo,t_alpha,realized_beta,ann_sharpe,n_months
portfolio,,,,,,,
VW_1,0.757,3.865,-0.155,-1.236,0.947,0.486,758
VW_2,0.903,5.543,0.076,0.870,0.860,0.697,758
VW_3,0.939,6.193,0.146,1.968,0.824,0.779,758
VW_4,1.041,6.905,0.222,3.507,0.851,0.869,758
VW_5,0.995,6.309,0.111,2.033,0.918,0.794,758
VW_6,1.079,6.295,0.101,2.004,1.016,0.792,758
VW_7,1.051,5.594,-0.017,-0.289,1.110,0.704,758
VW_8,1.001,4.808,-0.143,-1.730,1.188,0.605,758
VW_9,1.022,4.153,-0.305,-2.845,1.379,0.523,758



=== Value-weighted deciles sorted on 1y beta ===


,mean_ex_ret_%/mo,t_mean,alpha_%/mo,t_alpha,realized_beta,ann_sharpe,n_months
portfolio,,,,,,,
VW_1,0.620,3.446,-0.167,-1.359,0.824,0.435,752
VW_2,0.922,6.423,0.210,2.668,0.746,0.811,752
VW_3,0.891,6.094,0.147,1.969,0.778,0.770,752
VW_4,1.026,6.733,0.210,3.277,0.853,0.851,752
VW_5,1.149,7.098,0.261,4.406,0.929,0.897,752
VW_6,1.011,5.735,0.032,0.549,1.024,0.724,752
VW_7,1.012,5.121,-0.085,-1.282,1.148,0.647,752
VW_8,0.992,4.474,-0.217,-2.569,1.265,0.565,752
VW_9,1.077,4.208,-0.287,-2.600,1.428,0.532,752



=== Value-weighted deciles sorted on 2y beta ===


,mean_ex_ret_%/mo,t_mean,alpha_%/mo,t_alpha,realized_beta,ann_sharpe,n_months
portfolio,,,,,,,
VW_1,0.718,4.650,0.064,0.597,0.690,0.592,740
VW_2,0.807,5.770,0.162,1.902,0.680,0.735,740
VW_3,0.996,6.961,0.284,3.886,0.750,0.886,740
VW_4,1.103,7.222,0.307,4.675,0.839,0.920,740
VW_5,1.054,6.194,0.144,2.269,0.959,0.789,740
VW_6,0.964,5.261,-0.031,-0.512,1.049,0.670,740
VW_7,1.038,5.054,-0.075,-1.072,1.172,0.644,740
VW_8,1.029,4.569,-0.180,-2.192,1.274,0.582,740
VW_9,1.042,3.863,-0.348,-2.853,1.465,0.492,740



=== Value-weighted deciles sorted on 5y beta ===


,mean_ex_ret_%/mo,t_mean,alpha_%/mo,t_alpha,realized_beta,ann_sharpe,n_months
portfolio,,,,,,,
VW_1,0.894,6.526,0.388,3.655,0.533,0.838,728
VW_2,0.869,6.430,0.256,3.091,0.646,0.826,728
VW_3,1.034,7.010,0.314,4.106,0.759,0.900,728
VW_4,0.959,6.008,0.142,2.031,0.861,0.771,728
VW_5,1.017,5.786,0.093,1.390,0.974,0.743,728
VW_6,1.065,5.537,0.043,0.635,1.076,0.711,728
VW_7,0.923,4.378,-0.205,-2.904,1.189,0.562,728
VW_8,0.962,4.151,-0.269,-3.252,1.297,0.533,728
VW_9,1.069,3.846,-0.350,-2.842,1.495,0.494,728



=== Value-weighted deciles sorted on 10y beta ===


,mean_ex_ret_%/mo,t_mean,alpha_%/mo,t_alpha,realized_beta,ann_sharpe,n_months
portfolio,,,,,,,
VW_1,0.888,6.629,0.411,3.876,0.495,0.865,704
VW_2,0.830,5.916,0.218,2.409,0.636,0.772,704
VW_3,1.005,6.565,0.252,3.328,0.782,0.857,704
VW_4,1.066,6.195,0.195,2.551,0.904,0.809,704
VW_5,1.044,5.787,0.106,1.515,0.974,0.756,704
VW_6,0.963,4.885,-0.069,-0.943,1.072,0.638,704
VW_7,1.070,4.982,-0.050,-0.615,1.163,0.650,704
VW_8,0.992,4.231,-0.240,-2.818,1.279,0.552,704
VW_9,1.134,4.098,-0.273,-2.263,1.461,0.535,704


In [19]:
# Equal-weighted long-short across windows
pd.concat({k: v.loc["EW_HL"] for k, v in summaries.items()}, axis=1).T.round(3)

,mean_ex_ret_%/mo,t_mean,alpha_%/mo,t_alpha,realized_beta,ann_sharpe,n_months
6m,-0.086,-0.513,-0.562,-3.740,0.495,-0.065,758.0
1y,-0.002,-0.010,-0.701,-4.297,0.731,-0.001,752.0
2y,0.106,0.453,-0.766,-4.185,0.920,0.058,740.0
5y,0.045,0.176,-0.942,-4.907,1.040,0.023,728.0
10y,0.063,0.243,-0.950,-5.019,1.051,0.032,704.0


## 8. Quick plot: realized beta and alpha by decile (value-weighted)

In [20]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 2, figsize=(11, 4), sharex=True)
for name in WINDOWS:
    s = summaries[name].loc[[f"VW_{k}" for k in range(1, 11)]]
    axes[0].plot(range(1, 11), s["realized_beta"].values, marker="o", label=name)
    axes[1].plot(range(1, 11), s["alpha_%/mo"].values, marker="o", label=name)
axes[0].set(title="Realized beta by formation-beta decile", xlabel="Decile", ylabel="beta")
axes[1].set(title="CAPM alpha by decile (% / month)", xlabel="Decile")
axes[1].axhline(0, color="grey", lw=0.8)
axes[0].legend(title="window")
plt.tight_layout(); plt.show()

ModuleNotFoundError: No module named 'matplotlib'

## 9. Extension: 1-month beta (needs daily data)

Monthly data gives one observation per month, so a 1-month beta is undefined. To add it:

1. Load CRSP **daily** (`permno, date, ret`) and daily market and risk-free returns.
2. For each (permno, month), regress daily excess returns on daily market excess returns (require about 15 or more days; consider Dimson lags for thin trading).
3. Pivot the result to date x permno at month-end and pass it to `decile_sort()` exactly as in section 7.